# HydroServer in 40 Minutes

**Peer-to-Peer Technical Workshop Open Source and Interoperable Hydrological and Meteorological Data Systems for Multi-Hazard Early Warning Systems**

This notebook is a guided HydroServer walkthrough for hydrology, meteorology, forecasting, and early-warning professionals. You do not need to write Python from scratch. Run the cells in order and only edit values in the configuration cell when the facilitator asks you to.


## Workshop Flow

This notebook has two layers:

1. **Required live path:** configuration, connection, workspace lookup, metadata inspection, local observations, quality-control checks, plotting, and a safe loading demonstration.
2. **Optional reference sections:** broader HydroServerPy data management, authenticated workspace administration, datastream operations, `hydroserverpy.etl`, and HydroServer-backed quality control.

For the 1-hour workshop, run the required live path first. The optional reference sections are there so the notebook matches the HydroServerPy management guides and can remain useful after the session.


## Configuration

Only this cell is meant to be edited during the workshop. The default path does not require credentials. For the facilitator demo, connect with an API key or API key and use the already-created `hydroserver_uganda_demo` workspace. Workspace creation remains available as an optional fallback.


In [ ]:
from getpass import getpass
from pathlib import Path

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "hydroserver_uganda_demo"
WORKSPACE_IS_PRIVATE = False

# Choose one: "anonymous" or "api_key".
AUTH_METHOD = "anonymous"

# Keep secrets out of the notebook. Paste them only when prompted.
HYDROSERVER_API_KEY = ""

# Facilitator-only: keep False when the workspace already exists.
CREATE_WORKSPACE_IF_MISSING = False

# Facilitator-only: create an API key for the workspace after authenticating.
CREATE_WORKSPACE_API_KEY = False
WORKSPACE_API_KEY_NAME = "uganda-demo-api-key"

# Facilitator-only: create and later clean up demo metadata resources.
CREATE_DEMO_METADATA = False
DELETE_DEMO_RESOURCES_AT_END = False
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = ""  # Leave blank to generate a unique suffix for created demo resources.

# Generate a compact five-year daily historical time series for upload demonstrations.
GENERATE_FIVE_YEAR_OBSERVATIONS = True
FAKE_OBSERVATION_YEARS = 5

# Leave this False unless the facilitator has prepared credentials and a demo datastream.
ENABLE_LIVE_WRITE = False
DEMO_DATASTREAM_ID = ""  # Facilitator-provided datastream UUID for optional live write.

DATA_DIR_CANDIDATES = [Path("data"), Path("../data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), DATA_DIR_CANDIDATES[0])
STREAMFLOW_CSV = DATA_DIR / "sample_streamflow_observations.csv"
FORECAST_CSV = DATA_DIR / "sample_forecast_timeseries.csv"

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Demo workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")

## Connect to HydroServer

HydroServer can be used anonymously for public data. Creating workspaces, creating API keys, creating metadata, or uploading observations requires authentication. This notebook starts in anonymous mode, but it can also connect with an API key when the facilitator wants to demonstrate authenticated actions.


In [ ]:
def _prompt_if_needed(value, prompt):
    if value:
        return value
    try:
        return getpass(prompt)
    except Exception:
        return input(prompt)

try:
    from hydroserverpy import HydroServer

    if AUTH_METHOD == "api_key":
        api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print(f"Connected to {HYDROSERVER_HOST} with an API key.")
    elif AUTH_METHOD == "anonymous":
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print(f"Connected anonymously to {HYDROSERVER_HOST}.")
    else:
        raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")
except Exception as exc:
    hs_api = None
    print("HydroServerPy is unavailable or could not initialize.")
    print(f"Continuing with local sample data only. Details: {exc}")


## Track Created Resource UUIDs

The optional authenticated sections can create demo resources. This registry captures the object and UUID for each resource so later cells can show what was created, use those UUIDs smoothly in downstream examples, and clean up the resources at the end of the notebook.


In [ ]:
import pandas as pd
from uuid import UUID

created_resources = {
    "workspace": None,
    "thing": None,
    "observed_property": None,
    "unit": None,
    "sensor": None,
    "processing_level": None,
    "result_qualifier": None,
    "datastream": None,
    "orchestration_system": None,
    "data_connection": None,
    "task": None,
}

def resource_uid(resource):
    if resource is None:
        return None
    if isinstance(resource, UUID):
        return str(resource)
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            value = resource.get(key)
            if value:
                return str(value)
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None

def resource_name(resource):
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code"):
            value = resource.get(key)
            if value:
                return str(value)
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__

def record_created_resource(resource_type, resource):
    created_resources[resource_type] = resource
    print(f"{resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
    return resource

def created_resources_dataframe():
    rows = []
    for resource_type, resource in created_resources.items():
        if resource is not None:
            rows.append({
                "resource_type": resource_type,
                "name_or_code": resource_name(resource),
                "uuid": resource_uid(resource),
                "python_type": type(resource).__name__,
            })
    return pd.DataFrame(rows)

print("Created resource registry initialized.")


## Find or Optionally Create Demo Workspace

For this workshop, the demo workspace is named `hydroserver_uganda_demo` and is expected to already exist for the facilitator demo. This cell looks for that workspace when authenticated. If it does not exist, it only creates the workspace when `CREATE_WORKSPACE_IF_MISSING = True`. If the notebook is running anonymously, it explains the limitation and continues without failing.


In [ ]:
workspace = None

def _collection_items(collection):
    return getattr(collection, "items", collection)


if hs_api is None:
    print("Skipping workspace setup because the HydroServer client is not available.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot find or create workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'api_key' to use the already-created workspace.")
else:
    try:
        workspaces = _collection_items(hs_api.workspaces.list(fetch_all=True))
        workspace = next((item for item in workspaces if getattr(item, "name", None) == WORKSPACE_NAME), None)
        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE)
            record_created_resource("workspace", workspace)
            print(f"Created workspace: {WORKSPACE_NAME}")
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found.")
            print("Ask the facilitator to create it first, or set CREATE_WORKSPACE_IF_MISSING = True for a creation demo.")
        else:
            print(f"Using existing workspace: {WORKSPACE_NAME}")
            print(f"Workspace UUID: {resource_uid(workspace)}")
    except Exception as exc:
        print(f"Could not create or find workspace '{WORKSPACE_NAME}': {exc}")
        print("The rest of the notebook can continue with local sample data.")


## Optional: Create a Workspace API Key

Anonymous mode is useful for public reads, but it is limited: it cannot create a workspace, create metadata, upload observations, or create API keys. If the facilitator wants to show a complete authenticated workflow, first connect with an API key, create or reuse `hydroserver_uganda_demo`, and then set `CREATE_WORKSPACE_API_KEY = True`.

API key secrets are shown once. Do not save them in this notebook.


In [ ]:
workspace_api_key = None
workspace_api_key_secret = None

if not CREATE_WORKSPACE_API_KEY:
    print("Skipping workspace API key creation because CREATE_WORKSPACE_API_KEY is False.")
elif AUTH_METHOD == "anonymous":
    print("Switch AUTH_METHOD to 'api_key' before creating a workspace API key.")
elif hs_api is None or workspace is None:
    print("Skipping API key creation because an authenticated workspace is not available.")
else:
    try:
        api_key_roles = _collection_items(
            hs_api.roles.list(workspace=workspace, is_apikey_role=True, fetch_all=True)
        )
        if not api_key_roles:
            raise RuntimeError("No API-key roles were returned for this workspace.")

        workspace_api_key, workspace_api_key_secret = workspace.create_api_key(
            name=WORKSPACE_API_KEY_NAME,
            role=api_key_roles[0],
        )
        print(f"Created API key metadata: {getattr(workspace_api_key, 'name', WORKSPACE_API_KEY_NAME)}")
        print("Copy this API key secret now. It will not be shown again:")
        print(workspace_api_key_secret)
    except Exception as exc:
        print(f"Could not create a workspace API key: {exc}")


## HydroServerPy Data Management Essentials

HydroServerPy exposes HydroServer resources through properties on the connection object. Most managed resources follow the same pattern:

- `list(...)`: retrieve a collection of resources.
- `get(uid=...)`: retrieve one resource by ID.
- `create(...)`: create a resource when authenticated and authorized.
- Returned collections expose records through `.items`.

Core resource properties include `workspaces`, `things`, `datastreams`, `sensors`, `units`, `processinglevels`, `observedproperties`, `resultqualifiers`, `orchestrationsystems`, `dataconnections`, and `tasks`.

| Resource property | Hydrologic meaning in this workshop |
|---|---|
| `workspaces` | Project or organization area that owns managed resources |
| `things` | Sites, gauges, stations, outlets, or forecast points |
| `datastreams` | Time series connecting a site, variable, sensor/method, unit, and processing level |
| `sensors` | Field instruments, models, data sources, or methods |
| `units` | Measurement units such as m3/s, mm, m, degC |
| `processinglevels` | Raw, quality-controlled, modeled, or otherwise processed data states |
| `observedproperties` | Streamflow, rainfall, water level, temperature, forecast value |
| `resultqualifiers` | Flags such as estimated, suspect, provisional, or power failure |
| `orchestrationsystems` | Systems that run scheduled data loading or processing jobs |
| `dataconnections` | Extract-transform-load connection definitions |
| `tasks` | Scheduled or manual ETL task configurations |


In [ ]:
core_resources = [
    "workspaces",
    "things",
    "datastreams",
    "sensors",
    "units",
    "processinglevels",
    "observedproperties",
    "resultqualifiers",
    "orchestrationsystems",
    "dataconnections",
    "tasks",
]

for resource_property in core_resources:
    print(resource_property, "->", hasattr(hs_api, resource_property) if hs_api is not None else "client unavailable")


### Collections, Pagination, Ordering, and Filtering

Many HydroServer endpoints return collections. The common read pattern is: call `list(...)`, then work with `collection.items`. You can request pages, order records, filter records, or fetch all pages. These examples are safe read-only patterns; if the network is unavailable, continue with the local CSV sections.

Common `list(...)` options include:

- `page`: page number, starting at 1.
- `page_size`: number of items per page.
- `order_by`: list of fields to order by, with `-` for descending order.
- `fetch_all=True`: retrieve all available pages.
- Resource filters such as `workspace`, `thing`, `bbox`, `tag`, `is_private`, and `is_associated`.


In [ ]:
if hs_api is None:
    print("Skipping collection examples because the HydroServer client is not available.")
else:
    try:
        print(f"Accessing workspaces collection...")
        workspaces_page = hs_api.workspaces.list(page_size=5, page=1, order_by=["name"])
        workspace_items = getattr(workspaces_page, "items", workspaces_page)
        print(f"First page returned {len(workspace_items)} workspaces.")
        for workspace_item in workspace_items[:5]:
            print("-", getattr(workspace_item, "name", "Unnamed workspace"))
    except Exception as exc:
        print(f"Could not read workspace collection: {exc}")

# # Additional read-only collection patterns:
# all_workspaces = hs_api.workspaces.list(fetch_all=True)
# public_workspaces = hs_api.workspaces.list(is_private=False)
# your_private_workspaces = hs_api.workspaces.list(is_private=True, is_associated=True)
# ordered_workspaces = hs_api.workspaces.list(order_by=["name"])
# descending_workspaces = hs_api.workspaces.list(order_by=["-name", "is_private"])
# next_page = workspaces_page.next_page()
# previous_page = workspaces_page.previous_page()
# full_collection = workspaces_page.fetch_all()
# workspace_things = hs_api.things.list(workspace="<workspace-uuid>")
# thing_datastreams = hs_api.datastreams.list(thing="<thing-uuid>")
# bounded_things = hs_api.things.list(bbox=(32.0, -1.5, 35.0, 4.5))
# tagged_things = hs_api.things.list(tag=("Region", "Uganda"))


### Workspace-Scoped Resources

Workspaces organize access to user-managed data. For this workshop, the intended workspace is `hydroserver_uganda_demo`. The normal facilitator path is to reuse an already-created workspace. Programmatic workspace creation remains optional and disabled by default.

Once a workspace exists, you can retrieve related resources either through workspace properties or through filtered service calls.


In [ ]:
workspace_uid = getattr(workspace, "uid", None) or getattr(workspace, "id", None)

if workspace is None:
    print("No authenticated workspace object is available yet.")
    print(f"The intended workspace for this workshop is '{WORKSPACE_NAME}'.")
else:
    print(f"Workspace ready: {getattr(workspace, 'name', WORKSPACE_NAME)}")
    print(f"Workspace ID: {workspace_uid}")
    print("Workspace-scoped resources can be listed with filters such as workspace=workspace_uid.")
    workspace_context = pd.DataFrame([{"resource_type": "workspace", "name": getattr(workspace, "name", WORKSPACE_NAME), "uuid": workspace_uid}]) if "pd" in globals() else None
    if workspace_context is not None:
        display(workspace_context)

# Workspace resource properties and filtered list examples:
# workspace_collaborators = workspace.collaborators
# workspace_api_keys = workspace.apikeys
# workspace_roles = workspace.roles
# workspace_things = workspace.things
# workspace_observed_properties = workspace.observedproperties
# workspace_units = workspace.units
# workspace_processing_levels = workspace.processinglevels
# workspace_sensors = workspace.sensors
# workspace_orchestration_systems = workspace.orchestrationsystems
# workspace_data_connections = workspace.dataconnections
# workspace_tasks = workspace.tasks
# workspace_things = hs_api.things.list(workspace=workspace_uid)
# workspace_units = hs_api.units.list(workspace=workspace_uid)
# workspace_observed_properties = hs_api.observedproperties.list(workspace=workspace_uid)
# workspace_datastreams = hs_api.datastreams.list(workspace=workspace_uid)

# Optional authenticated create/update examples. Keep disabled for the normal workshop path.
# new_workspace = hs_api.workspaces.create(name="hydroserver_uganda_demo", is_private=False)
# workspace.name = "New Workspace Name"
# workspace.is_private = True
# workspace.save()

# Workspace administration examples are intentionally omitted from the 40-minute workshop.


### Metadata Resource Create Patterns

The examples below show the shape of common HydroServer metadata creation calls. They are templates for authenticated facilitator work, not default workshop actions. Most metadata resources can also be modified with attribute assignment followed by `.save()`, refreshed with `.refresh()`, and removed with `.delete()`. Keep update/delete examples disabled unless you are working in a disposable demo workspace.


In [ ]:
from datetime import datetime, timezone

if DEMO_RUN_SUFFIX:
    demo_run_suffix = DEMO_RUN_SUFFIX
else:
    demo_run_suffix = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {demo_run_suffix}"

# Thing / site example
thing_create_template = {
    "name": f"{demo_resource_prefix} Gauge",
    "description": "Workshop site for streamflow observations.",
    "sampling_feature_type": "Site",
    "sampling_feature_code": f"UGANDA_DEMO_GAUGE_{demo_run_suffix}",
    "site_type": "Stream",
    "latitude": 0.3476,
    "longitude": 32.5825,
    "elevation_m": 1200.0,
    "elevation_datum": "EGM96",
    "state": "Central Region",
    "county": "Kampala",
    "country": "UG",
    "data_disclaimer": "Workshop demonstration data.",
    "is_private": False,
    "workspace": workspace_uid or "<workspace-uuid>",
}

# Observed property, unit, sensor, processing level, and result qualifier examples
observed_property_create_template = {
    "name": f"{demo_resource_prefix} Streamflow",
    "definition": "Water discharge in a river channel",
    "description": "Streamflow used in the HydroServer workshop.",
    "observed_property_type": "Hydrology",
    "code": f"Streamflow_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
unit_create_template = {
    "name": f"{demo_resource_prefix} Cubic meters per second",
    "symbol": "m3/s",
    "definition": "Cubic meters per second",
    "unit_type": "Discharge",
    "workspace": workspace_uid or "<workspace-uuid>",
}
sensor_create_template = {
    "name": f"{demo_resource_prefix} Gauge or Model",
    "description": "Sensor or data source used for workshop observations.",
    "encoding_type": "application/json",
    "manufacturer": "Workshop",
    "sensor_model": "Demo",
    "sensor_model_link": "https://playground.hydroserver.org",
    "method_type": "Sensor",
    "method_link": "https://playground.hydroserver.org",
    "method_code": f"WORKSHOP_DEMO_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
processing_level_create_template = {
    "code": f"RAW_{demo_run_suffix}",
    "definition": "Raw",
    "explanation": "Data have not been processed or quality controlled.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
result_qualifier_create_template = {
    "code": f"SUSPECT_{demo_run_suffix}",
    "description": "Observation should be reviewed before operational use.",
    "workspace": workspace_uid or "<workspace-uuid>",
}

print(f"Metadata create templates are ready for facilitator-led authenticated workflows. Run suffix: {demo_run_suffix}")

# Disabled authenticated create calls:
# new_thing = hs_api.things.create(**thing_create_template)
# new_observed_property = hs_api.observedproperties.create(**observed_property_create_template)
# new_unit = hs_api.units.create(**unit_create_template)
# new_sensor = hs_api.sensors.create(**sensor_create_template)
# new_processing_level = hs_api.processinglevels.create(**processing_level_create_template)
# new_result_qualifier = hs_api.resultqualifiers.create(**result_qualifier_create_template)

# Disabled object methods:
# thing.add_tag(key="Region", value="Uganda")
# thing.update_tag(key="Region", value="Lake Victoria Basin")
# thing.delete_tag(key="Region")
# thing.add_file_attachment(file=file_attachment, file_attachment_type="Photo")
# thing.delete_file_attachment(name="site_photo.png")
# thing_datastreams = thing.datastreams
# thing.refresh()
# thing.delete()


metadata_template_summary = pd.DataFrame([
    {"resource_type": "thing", "name_or_code": thing_create_template["name"], "workspace_uuid": thing_create_template["workspace"]},
    {"resource_type": "observed_property", "name_or_code": observed_property_create_template["code"], "workspace_uuid": observed_property_create_template["workspace"]},
    {"resource_type": "unit", "name_or_code": unit_create_template["symbol"], "workspace_uuid": unit_create_template["workspace"]},
    {"resource_type": "sensor", "name_or_code": sensor_create_template["method_code"], "workspace_uuid": sensor_create_template["workspace"]},
    {"resource_type": "processing_level", "name_or_code": processing_level_create_template["code"], "workspace_uuid": processing_level_create_template["workspace"]},
    {"resource_type": "result_qualifier", "name_or_code": result_qualifier_create_template["code"], "workspace_uuid": result_qualifier_create_template["workspace"]},
])
display(metadata_template_summary)

### Datastreams and Observations

A datastream is the time-series resource that links a thing/site, observed property, sensor/method, unit, processing level, and observations. The workshop's safe default is to prepare a HydroServer-ready observation table locally. Authenticated datastream creation, loading, replacing, and deletion remain facilitator-only.


In [ ]:
# Datastream create shape. Requires real resource UUIDs in an authenticated workspace.
datastream_create_template = {
    "name": f"{demo_resource_prefix} Streamflow",
    "description": "Workshop streamflow datastream.",
    "observation_type": "Field Observation",
    "sampled_medium": "Water",
    "no_data_value": -9999,
    "aggregation_statistic": "Continuous",
    "time_aggregation_interval": 1,
    "status": "Ongoing",
    "result_type": "Timeseries",
    "value_count": 0,
    "phenomenon_begin_time": datetime(year=2024, month=1, day=1),
    "phenomenon_end_time": None,
    "result_begin_time": datetime(year=2024, month=1, day=1),
    "result_end_time": None,
    "is_visible": True,
    "is_private": False,
    "thing": resource_uid(created_resources.get("thing")) or "<thing-uuid>",
    "sensor": resource_uid(created_resources.get("sensor")) or "<sensor-uuid>",
    "observed_property": resource_uid(created_resources.get("observed_property")) or "<observed-property-uuid>",
    "processing_level": resource_uid(created_resources.get("processing_level")) or "<processing-level-uuid>",
    "unit": resource_uid(created_resources.get("unit")) or "<unit-uuid>",
    "time_aggregation_interval_unit": "hours",
    "intended_time_spacing": 1,
    "intended_time_spacing_unit": "hours",
}

print("Datastream create template includes the required related resource IDs.")

# Authenticated datastream reference patterns. These are intentionally disabled in the default path.
# datastream = hs_api.datastreams.create(**datastream_create_template)
# workspace_datastreams = hs_api.datastreams.list(workspace=workspace_uid)
# thing_datastreams = hs_api.datastreams.list(thing=resource_uid(created_resources.get("thing")))
# datastream = hs_api.datastreams.get(uid="<datastream-uuid>")
# thing = datastream.thing
# sensor = datastream.sensor
# observed_property = datastream.observed_property
# unit = datastream.unit
# processing_level = datastream.processing_level
# observations_df = datastream.get_observations(
#     phenomenon_time_min=datetime(year=2023, month=1, day=1),
#     phenomenon_time_max=datetime(year=2023, month=12, day=31),
# ).dataframe
# full_observations_df = datastream.get_observations(fetch_all=True).dataframe

# Observation write examples. Keep disabled unless ENABLE_LIVE_WRITE is True and DEMO_DATASTREAM_ID is prepared.
# datastream.load_observations(hydroserver_observations.dropna())
# datastream.load_observations(hydroserver_observations.dropna(), mode="replace")
# observations_with_qualifiers = hydroserver_observations.assign(result_qualifier_codes=[[]] * len(hydroserver_observations))
# datastream.load_observations(observations_with_qualifiers)

# Disabled datastream administration examples:
# datastream.add_tag(key="MaxAllowableResult", value=100)
# datastream.update_tag(key="MaxAllowableResult", value=120)
# datastream.delete_tag(key="MaxAllowableResult")
# datastream.add_file_attachment(file=file_attachment, file_attachment_type="Photo")
# datastream.delete_file_attachment(name="datastream_photo.png")
# datastream.refresh()

datastream_template_summary = pd.DataFrame([
    {"field": key, "value": value} for key, value in datastream_create_template.items()
])
display(datastream_template_summary.head(20))


### Optional: Create Demo Metadata and Capture UUIDs

This cell creates the resource chain needed for a smooth authenticated demo: observed property, unit, sensor, processing level, result qualifier, thing, and datastream. It records every created UUID so the upload and cleanup cells can use the resources directly. Keep `CREATE_DEMO_METADATA = False` unless the facilitator is authenticated and working in a disposable demo workspace.


In [ ]:
if not CREATE_DEMO_METADATA:
    print("Skipping demo metadata creation because CREATE_DEMO_METADATA is False.")
elif hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping demo metadata creation because an authenticated workspace is required.")
else:
    try:
        observed_property = record_created_resource(
            "observed_property",
            hs_api.observedproperties.create(**observed_property_create_template),
        )
        unit = record_created_resource(
            "unit",
            hs_api.units.create(**unit_create_template),
        )
        sensor = record_created_resource(
            "sensor",
            hs_api.sensors.create(**sensor_create_template),
        )
        processing_level = record_created_resource(
            "processing_level",
            hs_api.processinglevels.create(**processing_level_create_template),
        )
        result_qualifier = record_created_resource(
            "result_qualifier",
            hs_api.resultqualifiers.create(**result_qualifier_create_template),
        )
        thing = record_created_resource(
            "thing",
            hs_api.things.create(**thing_create_template),
        )

        datastream_create_template.update({
            "thing": resource_uid(thing),
            "sensor": resource_uid(sensor),
            "observed_property": resource_uid(observed_property),
            "processing_level": resource_uid(processing_level),
            "unit": resource_uid(unit),
        })
        datastream = record_created_resource(
            "datastream",
            hs_api.datastreams.create(**datastream_create_template),
        )
        DEMO_DATASTREAM_ID = str(resource_uid(datastream))
        print(f"DEMO_DATASTREAM_ID set from created datastream: {DEMO_DATASTREAM_ID}")
        display(created_resources_dataframe())
    except Exception as exc:
        print(f"Could not create demo metadata resources: {exc}")


### Orchestration Systems, Data Connections, Tasks, and Task Runs

HydroServer can also store ETL automation metadata. These examples are advanced authenticated templates for facilitators who want to show how operational data loading is configured. The nested `extractor_settings`, `transformer_settings`, and `loader_settings` use the same JSON-style shape as the data-management application.


In [ ]:
orchestration_system_create_template = {
    "name": "Workshop Data Loader",
    "orchestration_system_type": "SDL",
    "workspace": workspace_uid or "<workspace-uuid>",
}

csv_data_connection_template = {
    "name": "Example CSV Data Connection",
    "data_connection_type": "ETL",
    "workspace": workspace_uid or "<workspace-uuid>",
    "extractor_type": "HTTP",
    "extractor_settings": {
        "sourceUri": "https://www.example.com/data.csv?site={site_code}",
        "placeholderVariables": [{"name": "site_code", "type": "perTask"}],
    },
    "transformer_type": "CSV",
    "transformer_settings": {
        "headerRow": 1,
        "dataStartRow": 2,
        "delimiter": ",",
        "identifierType": "name",
        "timestamp": {"key": "TIMESTAMP", "format": "ISO8601", "timezoneMode": "embeddedOffset"},
    },
    "loader_type": "HydroServer",
    "loader_settings": {},
}

json_data_connection_template = {
    "name": "USGS Instantaneous Values",
    "data_connection_type": "ETL",
    "workspace": workspace_uid or "<workspace-uuid>",
    "extractor_type": "HTTP",
    "extractor_settings": {
        "sourceUri": (
            "https://waterservices.usgs.gov/nwis/iv/"
            "?format=json&sites={site_code}&parameterCd={param_code}"
            "&startDT={start_date}&endDT={end_date}"
        ),
        "placeholderVariables": [
            {"name": "site_code", "type": "perTask"},
            {"name": "param_code", "type": "perTask"},
            {
                "name": "start_date",
                "type": "runTime",
                "runTimeValue": "latestObservationTimestamp",
                "timestamp": {
                    "format": "ISO8601",
                    "timezoneMode": "daylightSavings",
                    "timezone": "America/Denver",
                },
            },
            {
                "name": "end_date",
                "type": "runTime",
                "runTimeValue": "jobExecutionTime",
                "timestamp": {
                    "format": "ISO8601",
                    "timezoneMode": "daylightSavings",
                    "timezone": "America/Denver",
                },
            },
        ],
    },
    "transformer_type": "JSON",
    "transformer_settings": {
        "JMESPath": "value.timeSeries[].values[].value[]",
        "timestamp": {"key": "dateTime", "format": "ISO8601", "timezoneMode": "embeddedOffset"},
    },
    "loader_type": "HydroServer",
    "loader_settings": {},
}

task_create_template = {
    "name": "Example Task",
    "extractor_variables": {},
    "transformer_variables": {},
    "loader_variables": {},
    "workspace": workspace_uid or "<workspace-uuid>",
    "orchestration_system": "<orchestration-system-uuid>",
    "data_connection": "<data-connection-uuid>",
    "interval": 1,
    "interval_period": "days",
    "mappings": [{"sourceIdentifier": "streamflow", "paths": [{"targetIdentifier": "<datastream-uuid>"}]}],
}

print("Advanced ETL administration templates are available for authenticated facilitator workflows.")

# Disabled authenticated calls:
# new_orchestration_system = hs_api.orchestrationsystems.create(**orchestration_system_create_template)
# new_data_connection = hs_api.dataconnections.create(**csv_data_connection_template)
# new_json_data_connection = hs_api.dataconnections.create(**json_data_connection_template)
# new_task = hs_api.tasks.create(**task_create_template)
# task = hs_api.tasks.get(uid="<task-uuid>")
# task.run()
# task_runs = task.get_task_runs()
# task_run = task.create_task_run(status="SUCCESS", started_at=started_at, finished_at=finished_at, result={"message": "Task executed successfully."})
# task.update_task_run(uid="<task-run-uuid>", status="SUCCESS")
# task.delete_task_run(uid="<task-run-uuid>")


## Inspect HydroServer Metadata

HydroServer organizes time-series data around operational concepts:

- **Thing:** a site, station, gauge, forecast point, or monitored feature.
- **Observed property:** what is measured or modeled, such as streamflow, water level, rainfall, or air temperature.
- **Sensor:** the instrument, model, data source, or process that produced the value.
- **Unit:** the measurement unit, such as m3/s, mm, degC, or m.
- **Datastream:** the time series that connects a thing, observed property, sensor, unit, and observations.

The next cell tries a small public metadata read. If network access or public data changes cause this to fail, the workshop continues with local sample data.


In [ ]:
if hs_api is None:
    print("Skipping HydroServer metadata read because the client is not available.")
else:
    for label, endpoint_name in [
        ("things/sites", "things"),
        ("datastreams", "datastreams"),
        ("observed properties", "observedproperties"),
        ("units", "units"),
    ]:
        try:
            endpoint = getattr(hs_api, endpoint_name)
            collection = endpoint.list() if hasattr(endpoint, "list") else endpoint
            items = getattr(collection, "items", collection)
            total_count = getattr(collection, "total_count", None)
            count = total_count if total_count is not None else len(items)
            print(f"HydroServer returned {count} {label}.")
        except Exception as exc:
            print(f"Could not read {label}: {exc}")


## Prepare Sample Observations

The bundled CSV gives a small table for teaching quality-control concepts. For upload demonstrations, the notebook generates a compact fake historical daily time series covering five years. That keeps the workshop realistic without storing a large static file in the repository.


In [ ]:
import math
import pandas as pd

observations = pd.read_csv(STREAMFLOW_CSV)
observations["timestamp"] = pd.to_datetime(observations["timestamp"], utc=True)

def generate_fake_historical_observations(years=5, end_date="2026-01-01"):
    end_timestamp = pd.Timestamp(end_date, tz="UTC")
    start_timestamp = end_timestamp - pd.DateOffset(years=years)
    timestamps = pd.date_range(start=start_timestamp, end=end_timestamp, freq="D", tz="UTC")
    rows = []
    for index, timestamp in enumerate(timestamps):
        seasonal = 35 + 18 * math.sin(2 * math.pi * index / 365.25)
        short_cycle = 4 * math.sin(2 * math.pi * index / 31)
        event_pulse = 22 if index % 179 in (0, 1, 2) else 0
        value = round(max(0.1, seasonal + short_cycle + event_pulse), 2)
        rows.append({
            "timestamp": timestamp,
            "value": value,
            "quality_note": "synthetic historical daily flow",
        })
    return pd.DataFrame(rows)

historical_observations = generate_fake_historical_observations(FAKE_OBSERVATION_YEARS)

print(f"Bundled QC sample rows: {len(observations)}")
print(f"Generated historical rows for upload demonstration: {len(historical_observations)}")
display(observations.head())
display(historical_observations.head())
display(historical_observations.tail())


In [ ]:
upload_source_observations = historical_observations if GENERATE_FIVE_YEAR_OBSERVATIONS else observations

hydroserver_observations = upload_source_observations[["timestamp", "value"]].rename(
    columns={"timestamp": "phenomenon_time", "value": "result"}
)
hydroserver_observations["phenomenon_time"] = pd.to_datetime(hydroserver_observations["phenomenon_time"], utc=True)

print(f"HydroServer upload payload rows: {len(hydroserver_observations)}")
print(f"Date range: {hydroserver_observations['phenomenon_time'].min()} to {hydroserver_observations['phenomenon_time'].max()}")
display(hydroserver_observations.head())
display(hydroserver_observations.tail())
display(hydroserver_observations.describe(include="all"))


## Quality Control and Forecast Readiness

Before data supports an early-warning workflow, we need fast checks for timestamp problems, missing values, suspicious magnitudes, and unit consistency. These checks are intentionally simple enough to explain in a few minutes.


In [ ]:
qc_summary = {
    "rows": len(observations),
    "missing_timestamps": int(observations["timestamp"].isna().sum()),
    "missing_values": int(observations["value"].isna().sum()),
    "unique_units": sorted(observations["unit"].dropna().unique().tolist()),
    "suspicious_high_values": int((observations["value"] > 100).sum()),
    "time_is_monotonic": bool(observations["timestamp"].is_monotonic_increasing),
}
qc_summary


In [ ]:
flagged = observations[
    observations["value"].isna() | (observations["value"] > 100)
].copy()
flagged[["timestamp", "value", "unit", "quality_note"]]


### Optional HydroServerPy Quality-Control Package

The local checks above are the live workshop path. HydroServerPy also includes a quality-control session object for HydroServer observations. This optional section shows the pattern for authenticated workflows: fetch observations from a prepared datastream, initialize `HydroServerQualityControl`, run checks such as `find_gaps`, inspect `hs_quality_control.observations`, and then upload quality-controlled data to a new datastream or processing level only when the facilitator has prepared that target.

Some HydroServer examples use account credentials, but this workshop keeps authentication to anonymous reads or `apikey=...` in `HydroServer(...)`.


In [ ]:
try:
    from hydroserverpy import HydroServerQualityControl
except Exception as exc:
    HydroServerQualityControl = None
    print(f"HydroServerQualityControl is not available in this environment: {exc}")

if not ENABLE_LIVE_WRITE:
    print("Skipping HydroServer-backed QC because ENABLE_LIVE_WRITE is False.")
    print("The local QC checks above remain the workshop default.")
elif hs_api is None or not DEMO_DATASTREAM_ID:
    print("Skipping HydroServer-backed QC because a client and DEMO_DATASTREAM_ID are required.")
elif HydroServerQualityControl is None:
    print("Skipping HydroServer-backed QC because HydroServerQualityControl could not be imported.")
else:
    try:
        datastream = hs_api.datastreams.get(uid=DEMO_DATASTREAM_ID)
        # Client versions differ on observation time filters. Verify parameter names before a live demo.
        observations_response = datastream.get_observations(
            include_quality=True,
            fetch_all=True,
        )
        observations_df = getattr(observations_response, "dataframe", observations_response)
        hs_quality_control = HydroServerQualityControl(
            datastream_id=datastream.uid,
            observations=observations_df,
        )
        hs_quality_control.find_gaps(time_value=15, time_unit="m")
        quality_controlled_observations = hs_quality_control.observations
        display(quality_controlled_observations.head())
    except Exception as exc:
        print(f"Could not run HydroServer-backed QC: {exc}")

# Authenticated QC upload pattern. Keep disabled unless a new target datastream and processing level are prepared.
# quality_controlled_datastream = hs_api.datastreams.get(uid="<quality-controlled-datastream-uuid>")
# quality_controlled_datastream.load_observations(quality_controlled_observations)


## Visualize the Time Series

A quick hydrograph is often the fastest way to find problems. Here the missing value and spike are visible before the data is used in a warning or forecast workflow.


In [ ]:
import matplotlib.pyplot as plt

ax = observations.plot(
    x="timestamp",
    y="value",
    marker="o",
    figsize=(9, 4),
    legend=False,
)
ax.set_title("Demo River Gauge Streamflow")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Streamflow (m3/s)")
ax.grid(True, alpha=0.3)
plt.show()


## Loading Demonstration

HydroServerPy can load observations from a pandas DataFrame into a datastream. The documented upload shape uses `phenomenon_time` and `result`, with optional `result_qualifier_codes` when quality flags are needed. For this workshop, the safe default is to prepare and inspect the payload. The optional live write cell only runs when the facilitator has provided a demo datastream and changed `ENABLE_LIVE_WRITE` to `True`.


In [ ]:
payload_preview = hydroserver_observations.dropna().copy()
print(f"Rows ready to upload: {len(payload_preview)}")
print(f"Payload columns: {list(payload_preview.columns)}")
display(payload_preview.head())
display(payload_preview.tail())

if created_resources.get("datastream") is not None:
    uploaded_target_summary = pd.DataFrame([{
        "resource_type": "datastream",
        "name": resource_name(created_resources["datastream"]),
        "uuid": resource_uid(created_resources["datastream"]),
        "rows_ready": len(payload_preview),
    }])
    display(uploaded_target_summary)
else:
    print("No created datastream is registered yet. Provide DEMO_DATASTREAM_ID or enable CREATE_DEMO_METADATA for a live upload.")


In [ ]:
if ENABLE_LIVE_WRITE:
    if hs_api is None:
        print("Cannot write because the HydroServer client is unavailable.")
    elif not DEMO_DATASTREAM_ID:
        print("Set DEMO_DATASTREAM_ID to a prepared demo datastream before enabling live writes.")
    else:
        datastream = created_resources.get("datastream") or hs_api.datastreams.get(uid=DEMO_DATASTREAM_ID)
        datastream.load_observations(payload_preview)
        print("Posted observations to the configured HydroServer datastream.")
        print(f"Loaded rows: {len(payload_preview)}")
        print(f"Target datastream UUID: {resource_uid(datastream) or DEMO_DATASTREAM_ID}")
        display(pd.DataFrame([{
            "datastream_name": resource_name(datastream),
            "datastream_uuid": resource_uid(datastream) or DEMO_DATASTREAM_ID,
            "loaded_rows": len(payload_preview),
            "first_time": payload_preview["phenomenon_time"].min(),
            "last_time": payload_preview["phenomenon_time"].max(),
        }]))
else:
    print("Live write is disabled. This is the safe default for the workshop.")
    print("The payload above is the table that would be posted to a prepared datastream.")


## Optional `hydroserverpy.etl` Package Overview

The `hydroserverpy.etl` package is for repeatable extract-transform-load pipelines. A pipeline has three parts:

- **Extractor:** retrieves raw data from HTTP, FTP, or a local file.
- **Transformer:** parses the raw payload into standardized observation rows.
- **Loader:** writes transformed observations to HydroServer datastreams.

For the live workshop, the direct pandas payload above is easier to teach. This section is a reference for operational workflows after the session.


In [ ]:
from hydroserverpy.etl import ETLPipeline
from hydroserverpy.etl.extractors import FTPExtractor, HTTPExtractor, LocalFileExtractor
from hydroserverpy.etl.loaders import HydroServerLoader
from hydroserverpy.etl.operations import (
    ArithmeticExpressionOperation,
    RatingCurveDataOperation,
    TemporalAggregationOperation,
)
from hydroserverpy.etl.transformers import (
    CSVTransformer,
    ETLDataMapping,
    ETLTargetPath,
    JSONTransformer,
)

local_extractor = LocalFileExtractor(source_uri=str(STREAMFLOW_CSV))
http_extractor_template = HTTPExtractor(source_uri="https://api.example.com/data/{station_id}/export.csv")
ftp_extractor_template = FTPExtractor(
    host="ftp.example.com",
    filepath="/data/{station_id}/readings.csv",
    port=21,
)

print("Extractor examples created: LocalFileExtractor, HTTPExtractor, FTPExtractor")


### ETL Transformers and Timestamp Configuration

Transformers parse source records. `CSVTransformer` and `JSONTransformer` both need a timestamp key. Timestamp handling controls how source times are parsed and normalized to UTC.

| Field | Purpose |
|---|---|
| `timestamp_type="iso"` | Parse standard ISO 8601 timestamps. |
| `timestamp_type="custom"` | Parse timestamps with `timestamp_format`. |
| `timezone_type=None` | Use embedded timestamp offsets, falling back to UTC for naive times. |
| `timezone_type="utc"` | Treat naive timestamps as UTC. |
| `timezone_type="offset"` | Apply a fixed offset such as `-0700` or `-07:00`. |
| `timezone_type="iana"` | Apply an IANA timezone such as `Africa/Kampala` or `America/Denver`. |


In [ ]:
csv_transformer = CSVTransformer(
    timestamp_key="timestamp",
    delimiter=",",
    header_row=1,
    data_start_row=2,
)

csv_transformer_by_index = CSVTransformer(
    timestamp_key="1",
    identifier_type="index",
    data_start_row=2,
)

custom_timestamp_transformer = CSVTransformer(
    timestamp_key="datetime",
    timestamp_type="custom",
    timestamp_format="%m/%d/%Y %H:%M:%S",
    timezone_type="utc",
)

offset_transformer = CSVTransformer(
    timestamp_key="datetime",
    timezone_type="offset",
    timezone="-0700",
)

iana_transformer = CSVTransformer(
    timestamp_key="datetime",
    timezone_type="iana",
    timezone="Africa/Kampala",
)

json_transformer = JSONTransformer(
    timestamp_key="timestamp",
    jmespath="response.data",
)

print("Transformer examples created with ISO, custom, UTC, offset, and IANA timezone settings.")


### ETL Data Mappings and Operations

Data mappings connect source identifiers to HydroServer datastream IDs. Each target path can include data operations. Operations run in order, so temporal aggregation should usually be last because it changes the data shape.

Supported examples include arithmetic expressions, rating curves, and temporal aggregation. Temporal aggregation supports `simple_mean`, `time_weighted_mean`, and `last_value_of_period`; windows are aligned to UTC or the configured timezone, and days with no observations are omitted.


In [ ]:
demo_target_id = DEMO_DATASTREAM_ID or "<datastream-uuid>"

data_mappings = [
    ETLDataMapping(
        source_identifier="value",
        target_paths=[ETLTargetPath(target_identifier=demo_target_id)],
    )
]

fahrenheit_to_celsius_path = ETLTargetPath(
    target_identifier=demo_target_id,
    data_operations=[
        ArithmeticExpressionOperation(
            expression="(x - 32) / 1.8",
            target_identifier=demo_target_id,
        )
    ],
)

rating_curve_path = ETLTargetPath(
    target_identifier=demo_target_id,
    data_operations=[
        RatingCurveDataOperation(
            rating_curve_url="https://example.com/curves/stage-discharge.csv",
            target_identifier=demo_target_id,
        )
    ],
)

daily_mean_path = ETLTargetPath(
    target_identifier=demo_target_id,
    data_operations=[
        TemporalAggregationOperation(
            aggregation_statistic="simple_mean",
            aggregation_interval=1,
            aggregation_interval_unit="day",
            timezone_type="iana",
            timezone="Africa/Kampala",
            target_identifier=demo_target_id,
        )
    ],
)

print("ETLDataMapping and ETLTargetPath examples are ready.")


### ETL Loader, Pipeline, Results, and Debugging

`HydroServerLoader` writes transformed observations to HydroServer. The cell below only assembles a pipeline when a HydroServer client is available. Running the pipeline is gated behind `ENABLE_LIVE_WRITE` and a prepared `DEMO_DATASTREAM_ID`.

When running production ETL, `raise_on_error=False` captures failures in the returned context. Inspect `context.status`, `context.stage`, `context.results`, and `context.results.target_results` to debug extract, transform, or load issues.


In [ ]:
if hs_api is None:
    etl_pipeline = None
    print("Skipping ETL pipeline assembly because the HydroServer client is not available.")
else:
    loader = HydroServerLoader(client=hs_api, chunk_size=5000)
    etl_pipeline = ETLPipeline(
        extractor=local_extractor,
        transformer=csv_transformer,
        loader=loader,
    )
    print("ETLPipeline assembled with LocalFileExtractor, CSVTransformer, and HydroServerLoader.")

if ENABLE_LIVE_WRITE and etl_pipeline is not None and DEMO_DATASTREAM_ID:
    context = etl_pipeline.run(
        data_mappings=data_mappings,
        raise_on_error=False,
    )
    print(context.status)
    print(context.stage)
    if context.results is not None:
        print(context.results.success_count)
        print(context.results.failure_count)
        print(context.results.skipped_count)
        print(context.results.values_loaded_total)
        for target_id, target in context.results.target_results.items():
            print(target_id, target.status, getattr(target, "values_loaded", None))
    if getattr(context, "error", None):
        print(context.error)
        print(context.traceback)
else:
    print("ETL run skipped. Set ENABLE_LIVE_WRITE=True and DEMO_DATASTREAM_ID to run against a prepared datastream.")

# Common ETL debugging categories:
# Extract stage: source not found, timeout, authentication failed, empty source, missing runtime variables.
# Transform stage: timestamp column not found, empty CSV, invalid JSON, JMESPath returned no records, invalid arithmetic expression.
# Load stage: missing datastream IDs, authentication failure, network failure, datastream lookup failure.


## Optional Forecast Extension

Early-warning systems often combine observations with forecast time series. The full Global Forecast Validation workflow is outside the core session, but the same HydroServer pattern applies: convert forecast output into timestamped values, check metadata and units, then publish or share through interoperable services.


In [ ]:
forecast = pd.read_csv(FORECAST_CSV)
forecast["timestamp"] = pd.to_datetime(forecast["timestamp"], utc=True)
forecast["forecast_issue_time"] = pd.to_datetime(forecast["forecast_issue_time"], utc=True)
forecast


In [ ]:
forecast_payload = forecast[["timestamp", "value"]].rename(
    columns={"timestamp": "phenomenon_time", "value": "result"}
).copy()
forecast_payload.head()


In [ ]:
ax = forecast.plot(
    x="timestamp",
    y="value",
    marker="o",
    figsize=(9, 4),
    legend=False,
)
ax.set_title("Demo Forecast Point Streamflow Forecast")
ax.set_xlabel("Forecast valid time (UTC)")
ax.set_ylabel("Forecast streamflow (m3/s)")
ax.grid(True, alpha=0.3)
plt.show()


## Optional Cleanup: Delete Demo-Created Resources

Run this section at the end of a facilitator-led authenticated demo if you created disposable resources during the notebook. The cleanup order deletes dependent resources first: tasks, data connections, orchestration systems, datastreams, things, result qualifiers, processing levels, sensors, units, observed properties, and finally an optionally created workspace.

Leave `DELETE_DEMO_RESOURCES_AT_END = False` unless you are sure the recorded resources are disposable workshop resources.


In [ ]:
cleanup_order = [
    "task",
    "data_connection",
    "orchestration_system",
    "datastream",
    "thing",
    "result_qualifier",
    "processing_level",
    "sensor",
    "unit",
    "observed_property",
    "workspace",
]

cleanup_preview = created_resources_dataframe()
if isinstance(cleanup_preview, pd.DataFrame) and not cleanup_preview.empty:
    display(cleanup_preview)
else:
    print("No resources are recorded in the created resource registry.")

if not DELETE_DEMO_RESOURCES_AT_END:
    print("Cleanup skipped because DELETE_DEMO_RESOURCES_AT_END is False.")
elif hs_api is None:
    print("Cleanup skipped because the HydroServer client is unavailable.")
else:
    for resource_type in cleanup_order:
        resource = created_resources.get(resource_type)
        if resource is None:
            continue
        try:
            print(f"Deleting {resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
            resource.delete()
            created_resources[resource_type] = None
        except Exception as exc:
            print(f"Could not delete {resource_type} {resource_uid(resource)}: {exc}")
    print("Cleanup finished. Remaining recorded resources:")
    remaining = created_resources_dataframe()
    if isinstance(remaining, pd.DataFrame) and not remaining.empty:
        display(remaining)
    else:
        print("None")


## Recap

In this session, we used HydroServer concepts to connect metadata, observations, quality checks, visualization, and forecast-ready time series. For operational systems, the same pattern scales to scheduled data loaders, public APIs, dashboards, and model input/output workflows.
